In [1]:
import pandas as pd
import numpy as np
import joblib


LOAD ALL MODELS

In [2]:
base = r"D:\ML\SM Modeling\AI_ENGINE_V3\models"

# resolution
res_model = joblib.load(base + r"\resolution_model.pkl")
res_tfidf = joblib.load(base + r"\res_tfidf.pkl")
res_svd = joblib.load(base + r"\res_svd.pkl")

# subcategory
sub_model = joblib.load(base + r"\sub_model.pkl")
sub_tfidf = joblib.load(base + r"\sub_tfidf.pkl")
sub_svd = joblib.load(base + r"\sub_svd.pkl")

# fault
fault_model = joblib.load(base + r"\fault_model.pkl")
fault_tfidf = joblib.load(base + r"\fault_tfidf.pkl")
fault_svd = joblib.load(base + r"\fault_svd.pkl")

print("All models loaded successfully")


All models loaded successfully


MASTER PREDICTION FUNCTION (CORE)

In [3]:
def predict_codes(short_desc, close_notes, task_text):

    text = f"{short_desc} {close_notes} {task_text}"

    # ---------------- RESOLUTION ----------------
    x = res_tfidf.transform([text])
    x = res_svd.transform(x)

    probs = res_model.predict_proba(x)[0]
    idx = np.argmax(probs)
    res_conf = probs[idx]
    res_pred = res_model.classes_[idx]

    # ---------------- SUBCATEGORY ----------------
    sub_text = text + " RESOLUTION_" + res_pred

    x = sub_tfidf.transform([sub_text])
    x = sub_svd.transform(x)

    probs = sub_model.predict_proba(x)[0]
    idx = np.argmax(probs)
    sub_conf = probs[idx]
    sub_pred = sub_model.classes_[idx]

    # ---------------- FAULT ----------------
    fault_text = sub_text + " SUBCAT_" + sub_pred

    x = fault_tfidf.transform([fault_text])
    x = fault_svd.transform(x)

    probs = fault_model.predict_proba(x)[0]
    idx = np.argmax(probs)
    fault_conf = probs[idx]
    fault_pred = fault_model.classes_[idx]

    # ---------------- FINAL CONFIDENCE ----------------
    final_conf = (res_conf*0.4 + sub_conf*0.3 + fault_conf*0.3)

    if final_conf >= 0.80:
        decision = "AUTO"
    else:
        decision = "NEEDS REVIEW"

    return {
        "Resolution Code": res_pred,
        "Subcategory": sub_pred,
        "Fault Category": fault_pred,
        "Confidence %": round(final_conf*100,2),
        "Decision": decision
    }


TEST ON REAL INCIDENT

In [4]:
out = predict_codes(
    "Link down for customer",
    "Service restored after reset",
    "Router rebooted and link up"
)

print(out)


{'Resolution Code': np.str_('Internal'), 'Subcategory': np.str_('Provisioning'), 'Fault Category': np.str_('RWT1 (Engineer Visit - Right When Tested )'), 'Confidence %': np.float64(36.26), 'Decision': 'NEEDS REVIEW'}
